# Minimal warehouse: per-line sales, picked options, one dashboard

This is the **minimal version** of the pizza shop warehouse, agreed with the teacher, after the second review:

- **An order is a header, its lines are the body. The fact is per line.** `FCT_ORDER_LINE` is the single atomic
  sales fact. Order-level money (commission, delivery fee) is *allocated down* to the lines, so nothing is lost and
  every line total still adds up to its order. The header lives on as the degenerate `order_number`.
- **Modifiers are options picked from a list, not free text.** The POS shows a fixed pick list per item
  (`DIM_MODIFIER_OPTION`: Cheese, Spice, Crust…) and records each pick as a row. Anything not on the list is caught
  by the load and parked on the *Invalid* row, so every add-on can be counted and grouped.

```
POS + Grab ──▶ stg ──▶ dw: 2 facts + 6 dimensions ──▶ agg: 3 tables ──▶ Owner dashboard
```

| Fact table | Grain: one row = | Feeds |
|---|---|---|
| `FCT_ORDER_LINE` | one menu item on one order (order money allocated to it) | `mart_channel_daily`, `mart_sales_daily` |
| `FCT_ORDER_LINE_OPTION` | one picked option on one order line | `mart_option_daily` |

The design still follows MADT8102 chapter 5: `_key` surrogate keys, special-case rows instead of NULL keys,
a junk dimension for order status, degenerate order and line numbers, and drilling across instead of joining facts.

นี่คือ **เวอร์ชันย่อ** ของ warehouse ร้านพิซซ่า หลังรีวิวรอบที่สอง:
- **ออร์เดอร์คือ header บรรทัดคือ body และ fact ต้องอยู่ระดับบรรทัด** `FCT_ORDER_LINE` คือ fact การขายตัวเดียวที่ระดับละเอียดที่สุด
  เงินระดับออร์เดอร์ (คอมมิชชัน ค่าส่ง) ถูก *กระจายลง* ไปที่บรรทัด ไม่มีอะไรหาย และยอดรวมของบรรทัดยังเท่ากับยอดของออร์เดอร์ ส่วน header เหลือเป็น `order_number` แบบ degenerate
- **modifier คือตัวเลือกที่กดเลือกจากรายการ ไม่ใช่ข้อความอิสระ** POS แสดงรายการตัวเลือกที่กำหนดไว้ของแต่ละเมนู (`DIM_MODIFIER_OPTION`: ชีส ความเผ็ด แป้ง…)
  และบันทึกทุกการเลือกเป็นแถว อะไรที่ไม่อยู่ในรายการจะถูกจับตอนโหลดและไปอยู่ที่แถว *Invalid* ของเพิ่มทุกอย่างจึงนับและจัดกลุ่มได้

In [1]:
import pandas as pd
import duckdb

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

con = duckdb.connect()                     # in-memory: gone when the kernel stops  |  อยู่ในหน่วยความจำ ปิด kernel แล้วหายไป
for schema in ["src", "stg", "dw", "agg"]: # sources, staging, the star, the aggregate tables
    con.execute(f"CREATE SCHEMA {schema}")  # ต้นทาง, staging, star, aggregate table

def q(sql):
    # run SQL and return a DataFrame so Jupyter shows it as a table  |  รัน SQL แล้วคืนเป็น DataFrame ให้ Jupyter แสดงเป็นตาราง
    return con.execute(sql).df()

def load(table, df):
    # copy a pandas DataFrame into a DuckDB table  |  คัดลอก DataFrame ของ pandas ลงเป็นตารางใน DuckDB
    con.register("_tmp", df)
    con.execute(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM _tmp")
    con.unregister("_tmp")

print("ready", duckdb.__version__)

ready 2.0.0


## 1 · Sources

Only the two sales sources: the **POS** (QR dine-in and phone) and a **delivery app export** (Grab here; LINE MAN,
ShopeeFood and Robinhood follow the same path). Two business days, Friday 4 and Saturday 5 Sep 2026. The shop trades
10:00–02:00, so the 01:20 order on the 5th belongs to Friday.

The POS keeps three tables: the order **header**, the order **lines** (the body), and the **options** picked on each
line. Options come from the pick list on the POS screen, so they arrive as codes, one row per pick.

มีแค่สองแหล่งข้อมูลการขาย: **POS** (QR และโทรศัพท์) และ **ไฟล์ export ของแอปเดลิเวอรี** ข้อมูลสองวันทำการ ศุกร์ 4 และเสาร์ 5 ก.ย. 2026
POS เก็บสามตาราง: **header** ของออร์เดอร์, **บรรทัด** ของออร์เดอร์ (body) และ **ตัวเลือก** ที่กดเลือกในแต่ละบรรทัด
ตัวเลือกมาจากรายการบนหน้าจอ POS จึงมาเป็นรหัส หนึ่งแถวต่อหนึ่งการเลือก

In [2]:
# ---- POS header: one row per order, Bangkok time  |  header: หนึ่งแถวต่อออร์เดอร์ เวลากรุงเทพ ----
pos_orders = pd.DataFrame([
    ("P-1001", "QR",    "2026-09-04 12:10", "m-081"),
    ("P-1002", "PHONE", "2026-09-04 19:30", "m-092"),
    ("P-1003", "QR",    "2026-09-05 01:20", None),     # after midnight: Friday's business day  |  หลังเที่ยงคืน: นับเป็นวันศุกร์
    ("P-1004", "QR",    "2026-09-05 12:45", None),
    ("P-1005", "PHONE", "2026-09-05 20:05", "m-081"),
], columns=["pos_order_id", "channel", "ordered_at", "member_id"])

# ---- POS body: one row per menu item on the order  |  body: หนึ่งแถวต่อเมนูในออร์เดอร์ ----
pos_order_items = pd.DataFrame([
    ("P-1001", 1, "PEP-L",  1, 329),
    ("P-1001", 2, "COKE",   2,  35),
    ("P-1002", 1, "HAW-L",  1, 319),
    ("P-1002", 2, "GARLIC", 1,  89),
    ("P-1003", 1, "PEP-L",  1, 329),
    ("P-1004", 1, "PEP-L",  2, 349),                    # price rose to 349 on Saturday  |  วันเสาร์ขึ้นราคาเป็น 349
    ("P-1004", 2, "COKE",   2,  35),
    ("P-1005", 1, "PEP-M",  1, 249),
    ("P-1005", 2, "HAW-L",  1, 319),
], columns=["pos_order_id", "line_no", "item_id", "qty", "unit_price"])

# ---- POS options: one row per option PICKED from the screen's list, with the price charged
# ---- ตัวเลือก: หนึ่งแถวต่อหนึ่งตัวเลือกที่กดจากรายการบนหน้าจอ พร้อมราคาที่คิดจริง
pos_order_item_options = pd.DataFrame([
    ("P-1001", 1, "CHEESE_EXTRA", 40),
    ("P-1003", 1, "CHEESE_EXTRA", 40),
    ("P-1003", 1, "SPICE_NONE",    0),
    ("P-1005", 1, "CHEESE_EXTRA", 40),
], columns=["pos_order_id", "line_no", "option_code", "price_delta"])

# ---- Grab: one flat export, order fields repeated per item row, UTC  |  Grab: ไฟล์แบน ฟิลด์ออร์เดอร์ซ้ำทุกแถว เวลาเป็น UTC ----
grab_export = pd.DataFrame([
    ("GF-201", "2026-09-04T11:30:00Z", "Pepperoni Pizza (L)", 1, 379, 25, 146.4),
    ("GF-201", "2026-09-04T11:30:00Z", "Garlic Bread",        1, 109, 25, 146.4),
    ("GF-202", "2026-09-04T18:10:00Z", "Hawaiian Pizza (L)",  1, 369, 25, 110.7),   # 01:10 Bangkok, still Friday
    ("GF-203", "2026-09-05T06:00:00Z", "Pepperoni Pizza (L)", 2, 399, 25, 252.9),
    ("GF-203", "2026-09-05T06:00:00Z", "Coke",                1,  45, 25, 252.9),
], columns=["orderID", "orderTime", "itemName", "quantity", "itemPrice", "deliveryFee", "merchantCommission"])

for name, df in [("src.pos_orders", pos_orders), ("src.pos_order_items", pos_order_items),
                 ("src.pos_order_item_options", pos_order_item_options), ("src.grab_export", grab_export)]:
    load(name, df)
print("loaded", len(pos_orders), "POS orders,", len(pos_order_items), "lines,", len(pos_order_item_options),
      "picked options, and", grab_export.orderID.nunique(), "Grab orders")

loaded 5 POS orders, 9 lines, 4 picked options, and 3 Grab orders


## 2 · Dimensions

Six dimensions and one bridge. Every dimension has special-case rows **0 = Not applicable, 1 = Unknown,
2 = Invalid**, readable flag values, and its hierarchy:

| Dimension | Hierarchy / notes |
|---|---|
| `DIM_DATE` | day › month › quarter › year, on the business day |
| `DIM_CHANNEL` | channel › channel type (Dine-in / Delivery); own channel or aggregator; commission rate |
| `DIM_CUSTOMER` | members on own channels; row 0 for orders with no identified customer |
| `DIM_MENU_ITEM` | SKU › item › category; **SCD2** on price |
| `DIM_MODIFIER_OPTION` | **the POS pick list**: option › option group (Cheese, Spice, Crust); add / remove / substitute; list price |
| `DIM_ORDER_INFO` | junk dimension: order status, cancellation flag |
| `BRG_CHANNEL_MENU_ITEM` | platform item names → our SKU |

`DIM_MODIFIER_OPTION` is the answer to "how do we analyse free text?": there is no free text. The same table drives
the POS screen and the warehouse, so every pick has a code, a group and a type that can be counted.

dimension หกตัวกับ bridge หนึ่งตัว ทุก dimension มี special-case row **0 = ไม่เกี่ยวข้อง, 1 = ไม่ทราบ, 2 = ไม่ถูกต้อง**
`DIM_MODIFIER_OPTION` คือคำตอบของคำถาม "ข้อความอิสระจะวิเคราะห์ยังไง": ไม่มีข้อความอิสระเลย ตารางเดียวกันนี้ใช้ทั้งบนหน้าจอ POS และใน warehouse
ทุกการเลือกจึงมีรหัส กลุ่ม และประเภทที่นับได้

In [3]:
SPECIAL = [(0, "Not applicable"), (1, "Unknown"), (2, "Invalid")]

def with_special(rows, columns, member, fill=None):
    # prepend the three special-case rows  |  เพิ่ม special-case row สามแถวไว้ข้างหน้า
    fill = fill or {}
    special = [(k, *[fill.get(c, "n/a") for c in columns[1:-1]], label) for k, label in SPECIAL]
    return pd.DataFrame(special + [(*r, member) for r in rows], columns=columns)

con.execute('''
CREATE TABLE dw.dim_date AS
SELECT CAST(strftime(d, '%Y%m%d') AS INT)                      AS date_key,
       CAST(d AS DATE)                                         AS full_date,
       dayname(d)                                              AS day_name,
       CASE WHEN isodow(d) IN (6, 7) THEN 'Weekend' ELSE 'Weekday' END AS weekend_flag,
       monthname(d)                                            AS month_name,
       'Q' || quarter(d) || ' ' || year(d)                     AS quarter,
       year(d)                                                 AS year,
       'Day'                                                   AS row_type
FROM generate_series(DATE '2026-09-01', DATE '2026-09-30', INTERVAL 1 DAY) AS t(d)
UNION ALL
SELECT k, NULL, 'n/a', 'n/a', 'n/a', 'n/a', NULL, label FROM (VALUES (0, 'Not applicable'), (1, 'Unknown')) AS s(k, label)
''')

load("dw.dim_channel", with_special([
    (101, "QR_DINEIN", "QR dine-in", "Dine-in",  "Own channel",  0.0),
    (102, "PHONE",     "Phone",      "Dine-in",  "Own channel",  0.0),
    (103, "GRAB",      "Grab",       "Delivery", "Aggregator",  30.0),
], ["channel_key", "channel_code", "channel_name", "channel_type", "channel_ownership", "commission_rate_pct", "row_type"],
   "Channel", {"commission_rate_pct": 0.0}))

load("dw.dim_customer", with_special([(101, "m-081", "Member"), (102, "m-092", "Member")],
    ["customer_key", "customer_nk", "membership", "row_type"], "Customer"))

far, start, rise = pd.Timestamp("9999-12-31"), pd.Timestamp("2026-01-01"), pd.Timestamp("2026-09-05")
load("dw.dim_menu_item", with_special([
    (101, "PEP-M",  "Pepperoni M",  "pizza", "Medium", 249, start, far),
    (102, "PEP-L",  "Pepperoni L",  "pizza", "Large",  329, start, rise),   # SCD2: Friday's price  |  ราคาวันศุกร์
    (103, "HAW-L",  "Hawaiian L",   "pizza", "Large",  319, start, far),
    (104, "GARLIC", "Garlic Bread", "side",  "n/a",     89, start, far),
    (105, "COKE",   "Coke",         "drink", "n/a",     35, start, far),
    (106, "PEP-L",  "Pepperoni L",  "pizza", "Large",  349, rise,  far),    # SCD2: from Saturday  |  ตั้งแต่วันเสาร์
], ["menu_item_key", "menu_item_nk", "item_name", "category", "size_name", "base_price_thb", "valid_from", "valid_to", "row_type"],
   "Menu item", {"base_price_thb": None, "valid_from": start, "valid_to": far}))

# The POS pick list: what the cashier can tap, grouped the way the screen groups it
# รายการตัวเลือกบน POS: สิ่งที่แคชเชียร์กดได้ จัดกลุ่มแบบเดียวกับหน้าจอ
load("dw.dim_modifier_option", with_special([
    (101, "CHEESE_EXTRA", "Extra cheese", "Cheese", "add",        40, "pizza"),
    (102, "CHEESE_NONE",  "No cheese",    "Cheese", "remove",      0, "pizza"),
    (103, "SPICE_NONE",   "No chili",     "Spice",  "remove",      0, "pizza"),
    (104, "SPICE_EXTRA",  "Extra chili",  "Spice",  "add",         0, "pizza"),
    (105, "CRUST_THIN",   "Thin crust",   "Crust",  "substitute",  0, "pizza"),
], ["option_key", "option_code", "option_name", "option_group", "option_type", "list_price_delta_thb", "applies_to_category", "row_type"],
   "Option", {"list_price_delta_thb": 0}))

load("dw.dim_order_info", pd.DataFrame([                        # junk dimension  |  junk dimension
    (101, "completed", "Not cancelled"), (102, "cancelled", "Cancelled"),
], columns=["order_info_key", "order_status", "cancellation_flag"]))

load("dw.brg_channel_menu_item", pd.DataFrame([
    (103, "PEP-M", "Pepperoni Pizza (M)"), (103, "PEP-L", "Pepperoni Pizza (L)"), (103, "HAW-L", "Hawaiian Pizza (L)"),
    (103, "GARLIC", "Garlic Bread"), (103, "COKE", "Coke"),
], columns=["channel_key", "menu_item_nk", "platform_item_name"]))

q("SELECT * FROM dw.dim_modifier_option ORDER BY option_key")

,option_key,option_code,option_name,option_group,option_type,list_price_delta_thb,applies_to_category,row_type
0,0,n/a,n/a,n/a,n/a,0,n/a,Not applicable
1,1,n/a,n/a,n/a,n/a,0,n/a,Unknown
2,2,n/a,n/a,n/a,n/a,0,n/a,Invalid
3,101,CHEESE_EXTRA,Extra cheese,Cheese,add,40,pizza,Option
4,102,CHEESE_NONE,No cheese,Cheese,remove,0,pizza,Option
5,103,SPICE_NONE,No chili,Spice,remove,0,pizza,Option
6,104,SPICE_EXTRA,Extra chili,Spice,add,0,pizza,Option
7,105,CRUST_THIN,Thin crust,Crust,substitute,0,pizza,Option


## 3 · Staging: header and body into one shape

Staging turns both sources into the same three shapes: order **header**, order **lines**, picked **options**.
UTC becomes Bangkok time, Grab's repeated order fields are taken once, and Grab's item names are mapped to our SKU.

staging แปลงทั้งสองแหล่งให้เป็นสามรูปแบบเดียวกัน: **header** ของออร์เดอร์, **บรรทัด** และ **ตัวเลือก** ที่เลือก
UTC กลายเป็นเวลากรุงเทพ เอาฟิลด์ออร์เดอร์ที่ Grab ซ้ำมาครั้งเดียว และแปลงชื่อเมนูของ Grab เป็น SKU

In [4]:
con.execute('''
CREATE TABLE stg.order_header AS
SELECT CASE channel WHEN 'QR' THEN 'QR_DINEIN' ELSE channel END AS channel_code,
       pos_order_id AS order_number, CAST(ordered_at AS TIMESTAMP) AS order_ts, member_id AS customer_nk,
       0.0 AS delivery_fee_thb, 0.0 AS commission_amount_thb
FROM src.pos_orders
UNION ALL
SELECT 'GRAB', orderID, MAX(timezone('Asia/Bangkok', CAST(orderTime AS TIMESTAMPTZ))), NULL,
       MAX(deliveryFee), MAX(merchantCommission)                -- repeated per item: take it once  |  ซ้ำทุกแถว: เอาครั้งเดียว
FROM src.grab_export GROUP BY orderID
''')
con.execute('''
CREATE TABLE stg.order_line AS
SELECT i.pos_order_id AS order_number, i.line_no AS line_number, i.item_id AS menu_item_nk,
       i.qty AS quantity, i.unit_price AS unit_price_thb
FROM src.pos_order_items i
UNION ALL
SELECT g.orderID, row_number() OVER (PARTITION BY g.orderID ORDER BY g.itemName), m.menu_item_nk, g.quantity, g.itemPrice
FROM src.grab_export g
LEFT JOIN dw.brg_channel_menu_item m ON m.channel_key = 103 AND m.platform_item_name = g.itemName
''')
con.execute('''
CREATE TABLE stg.line_option AS
SELECT pos_order_id AS order_number, line_no AS line_number, option_code, 1 AS quantity, price_delta AS price_delta_thb
FROM src.pos_order_item_options
''')
display(q("SELECT * FROM stg.order_header ORDER BY order_ts"))

,channel_code,order_number,order_ts,customer_nk,delivery_fee_thb,commission_amount_thb
0,QR_DINEIN,P-1001,2026-09-04 12:10:00,m-081,0.0,0.0
1,GRAB,GF-201,2026-09-04 18:30:00,NaN,25.0,146.4
2,PHONE,P-1002,2026-09-04 19:30:00,m-092,0.0,0.0
3,GRAB,GF-202,2026-09-05 01:10:00,NaN,25.0,110.7
4,QR_DINEIN,P-1003,2026-09-05 01:20:00,NaN,0.0,0.0
5,QR_DINEIN,P-1004,2026-09-05 12:45:00,NaN,0.0,0.0
6,GRAB,GF-203,2026-09-05 13:00:00,NaN,25.0,252.9
7,PHONE,P-1005,2026-09-05 20:05:00,m-081,0.0,0.0


## 4 · The fact is per line: allocate the header down

The header carries money that belongs to the whole order: Grab's **commission** and **delivery fee**. Keeping a
separate order-grain fact for it would mean two sales facts at two grains. Instead, each order's commission is
**allocated to its lines in proportion to line gross** (items plus picked options), rounded to the satang, with any
rounding remainder on the order's last line. Then:

- every sales measure lives at one grain, the line, and rolls up to any order, channel, day or item;
- an order count is simply `COUNT(DISTINCT order_number)`;
- the allocation is **reconciled**: allocated commission summed per order must equal the header's, or the load stops.

header มีเงินที่เป็นของทั้งออร์เดอร์: **ค่าคอมมิชชัน** และ **ค่าส่ง** ของ Grab ถ้าแยกเป็น fact ระดับออร์เดอร์อีกตัว จะมี fact การขายสองตัวที่ grain ต่างกัน
เราจึง **กระจายค่าคอมมิชชันของแต่ละออร์เดอร์ลงไปที่บรรทัดตามสัดส่วนยอดขายของบรรทัด** (ราคาเมนูบวกตัวเลือก) ปัดเป็นสตางค์ และเศษจากการปัดไปไว้ที่บรรทัดสุดท้าย
ผลคือ measure การขายทุกตัวอยู่ที่ grain เดียวคือบรรทัด จำนวนออร์เดอร์คือ `COUNT(DISTINCT order_number)` และยอดที่กระจายต้อง **กระทบยอด** กับ header ได้ ไม่อย่างนั้นการโหลดจะหยุด

In [5]:
# the dimension keys every line of an order shares  |  key ที่ทุกบรรทัดของออร์เดอร์ใช้ร่วมกัน
con.execute('''
CREATE TABLE stg.order_keys AS
SELECT h.*, c.channel_key,
       CAST(strftime(h.order_ts - INTERVAL 10 HOUR, '%Y%m%d') AS INT) AS date_key,      -- business-day rule, once
       COALESCE(cu.customer_key, 0)                                  AS customer_key,  -- 0 = Not applicable, never NULL
       101                                                           AS order_info_key
FROM stg.order_header h
JOIN dw.dim_channel c ON c.channel_code = h.channel_code
LEFT JOIN dw.dim_customer cu ON cu.customer_nk = h.customer_nk
''')

# FCT_ORDER_LINE: one menu item on one order, with the header's money allocated to it
# FCT_ORDER_LINE: เมนูหนึ่งรายการในออร์เดอร์หนึ่งใบ พร้อมเงินระดับ header ที่กระจายลงมา
con.execute('''
CREATE TABLE dw.fct_order_line AS
WITH opt AS (SELECT order_number, line_number, SUM(quantity * price_delta_thb) AS options_thb
             FROM stg.line_option GROUP BY ALL),
lines AS (
    SELECT l.*, k.date_key, k.channel_key, k.customer_key, k.order_info_key,
           k.commission_amount_thb AS order_commission, k.delivery_fee_thb AS order_delivery_fee,
           l.quantity * l.unit_price_thb                           AS item_amount_thb,
           COALESCE(o.options_thb, 0)                              AS option_amount_thb,
           l.quantity * l.unit_price_thb + COALESCE(o.options_thb, 0) AS line_gross_thb
    FROM stg.order_line l
    JOIN stg.order_keys k USING (order_number)
    LEFT JOIN opt o USING (order_number, line_number)
),
shares AS (
    SELECT *, line_gross_thb / SUM(line_gross_thb) OVER (PARTITION BY order_number) AS share,
           line_number = MAX(line_number) OVER (PARTITION BY order_number)          AS is_last_line
    FROM lines
),
alloc AS (
    SELECT *, round(order_commission * share, 2)   AS c_rounded,
              round(order_delivery_fee * share, 2) AS f_rounded
    FROM shares
)
SELECT a.date_key, a.channel_key, a.customer_key,
       COALESCE(mi.menu_item_key, 1) AS menu_item_key,                     -- 1 = Unknown, never NULL
       a.order_info_key,
       a.order_number, a.line_number,                                      -- degenerate: the header and the line
       a.quantity, a.unit_price_thb,                                       -- price snapshot
       a.item_amount_thb, a.option_amount_thb, a.line_gross_thb,
       a.c_rounded + CASE WHEN a.is_last_line THEN a.order_commission - SUM(a.c_rounded) OVER (PARTITION BY a.order_number) ELSE 0 END
                                                                          AS allocated_commission_thb,
       a.f_rounded + CASE WHEN a.is_last_line THEN a.order_delivery_fee - SUM(a.f_rounded) OVER (PARTITION BY a.order_number) ELSE 0 END
                                                                          AS allocated_delivery_fee_thb,
       a.line_gross_thb - allocated_commission_thb                        AS line_net_revenue_thb
FROM alloc a
JOIN dw.dim_date d ON d.date_key = a.date_key
LEFT JOIN dw.dim_menu_item mi ON mi.menu_item_nk = a.menu_item_nk AND mi.row_type = 'Menu item'
      AND d.full_date >= mi.valid_from AND d.full_date < mi.valid_to
''')

# FCT_ORDER_LINE_OPTION: one picked option on one order line; codes not on the pick list -> 2 (Invalid)
# FCT_ORDER_LINE_OPTION: ตัวเลือกหนึ่งตัวบนบรรทัดหนึ่งบรรทัด รหัสที่ไม่อยู่ในรายการ -> 2 (Invalid)
con.execute('''
CREATE TABLE dw.fct_order_line_option AS
SELECT l.date_key, l.channel_key, l.menu_item_key, COALESCE(op.option_key, 2) AS option_key,
       s.order_number, s.line_number, s.quantity, s.price_delta_thb
FROM stg.line_option s
JOIN dw.fct_order_line l USING (order_number, line_number)
LEFT JOIN dw.dim_modifier_option op ON op.option_code = s.option_code
''')

# Reconciliation: the lines must add back up to every header  |  กระทบยอด: บรรทัดต้องรวมกลับได้เท่ากับทุก header
bad = q('''
SELECT k.order_number, k.commission_amount_thb, SUM(l.allocated_commission_thb) AS allocated
FROM stg.order_keys k JOIN dw.fct_order_line l USING (order_number)
GROUP BY ALL HAVING round(SUM(l.allocated_commission_thb), 2) <> round(k.commission_amount_thb, 2)
''')
assert bad.empty, bad
invalid = con.execute("SELECT count(*) FROM dw.fct_order_line_option WHERE option_key = 2").fetchone()[0]
print("allocation reconciles with every order header;", invalid, "option(s) not on the pick list")

q('''
SELECT order_number, line_number, menu_item_key, line_gross_thb, allocated_commission_thb,
       allocated_delivery_fee_thb, line_net_revenue_thb
FROM dw.fct_order_line WHERE channel_key = 103 ORDER BY order_number, line_number
''')

allocation reconciles with every order header; 0 option(s) not on the pick list


,order_number,line_number,menu_item_key,line_gross_thb,allocated_commission_thb,allocated_delivery_fee_thb,line_net_revenue_thb
0,GF-201,1,104,109.0,32.7,5.58,76.3
1,GF-201,2,102,379.0,113.7,19.42,265.3
2,GF-202,1,103,369.0,110.7,25.00,258.3
3,GF-203,1,105,45.0,13.5,1.33,31.5
4,GF-203,2,106,798.0,239.4,23.67,558.6


GF-201 shows the allocation: 146.4 THB of commission on a 488 THB order becomes 113.70 on the 379 THB pepperoni and
32.70 on the 109 THB garlic bread. Summed back by `order_number`, the lines give the header exactly.

GF-201 แสดงการกระจาย: ค่าคอมมิชชัน 146.4 บาทของออร์เดอร์ 488 บาท กลายเป็น 113.70 บนเปปเปอโรนี 379 บาท และ 32.70 บนขนมปังกระเทียม 109 บาท
รวมกลับตาม `order_number` ก็ได้เท่ากับ header พอดี

## 5 · Three aggregate tables

| Aggregate table | Grain: one row = | Built from | Dashboard tile |
|---|---|---|---|
| `mart_channel_daily` | business date × channel | `FCT_ORDER_LINE` | Net revenue by channel, commission paid, average net per order |
| `mart_sales_daily` | business date × channel × SKU | `FCT_ORDER_LINE` | Top items, units and sales by item |
| `mart_option_daily` | business date × channel × option | `FCT_ORDER_LINE_OPTION` | Add-on revenue by option group; extra cheese per pizza (drilled across with `mart_sales_daily`) |

Order counts come from `COUNT(DISTINCT order_number)` on the line fact; ratios such as average order value are
computed by the dashboard from the additive parts.

จำนวนออร์เดอร์มาจาก `COUNT(DISTINCT order_number)` บน fact ระดับบรรทัด อัตราส่วนอย่างยอดเฉลี่ยต่อออร์เดอร์ให้แดชบอร์ดคำนวณจากส่วนที่บวกได้

In [6]:
con.execute('''
CREATE TABLE agg.mart_channel_daily AS
SELECT d.full_date AS business_date, d.day_name, d.weekend_flag, c.channel_name, c.channel_type, c.channel_ownership,
       count(DISTINCT l.order_number)     AS orders,
       SUM(l.line_gross_thb)              AS gross_sales_thb,
       SUM(l.allocated_commission_thb)    AS commission_thb,
       SUM(l.line_net_revenue_thb)        AS net_revenue_thb
FROM dw.fct_order_line l
JOIN dw.dim_date d USING (date_key) JOIN dw.dim_channel c USING (channel_key)
JOIN dw.dim_order_info oi USING (order_info_key)
WHERE oi.cancellation_flag = 'Not cancelled'
GROUP BY ALL
''')
con.execute('''
CREATE TABLE agg.mart_sales_daily AS
SELECT d.full_date AS business_date, c.channel_name, mi.menu_item_nk, mi.item_name, mi.category, mi.size_name,
       SUM(l.quantity)                  AS units_sold,
       count(DISTINCT l.order_number)   AS orders,
       SUM(l.item_amount_thb)           AS item_sales_thb,
       SUM(l.line_net_revenue_thb)      AS net_revenue_thb
FROM dw.fct_order_line l
JOIN dw.dim_date d USING (date_key) JOIN dw.dim_channel c USING (channel_key)
JOIN dw.dim_menu_item mi USING (menu_item_key) JOIN dw.dim_order_info oi USING (order_info_key)
WHERE oi.cancellation_flag = 'Not cancelled'
GROUP BY ALL
''')
con.execute('''
CREATE TABLE agg.mart_option_daily AS
SELECT d.full_date AS business_date, c.channel_name, op.option_group, op.option_name, op.option_type, mi.category,
       SUM(o.quantity)                       AS times_picked,
       SUM(o.quantity * o.price_delta_thb)   AS option_revenue_thb
FROM dw.fct_order_line_option o
JOIN dw.dim_date d USING (date_key) JOIN dw.dim_channel c USING (channel_key)
JOIN dw.dim_modifier_option op USING (option_key) JOIN dw.dim_menu_item mi USING (menu_item_key)
GROUP BY ALL
''')
for t in ["mart_channel_daily", "mart_sales_daily", "mart_option_daily"]:
    print(f"\n{t}")
    display(q(f"SELECT * FROM agg.{t} ORDER BY ALL"))


mart_channel_daily


,business_date,day_name,weekend_flag,channel_name,channel_type,channel_ownership,orders,gross_sales_thb,commission_thb,net_revenue_thb
0,2026-09-04,Friday,Weekday,Grab,Delivery,Aggregator,2,857.0,257.1,599.9
1,2026-09-04,Friday,Weekday,Phone,Dine-in,Own channel,1,408.0,0.0,408.0
2,2026-09-04,Friday,Weekday,QR dine-in,Dine-in,Own channel,2,808.0,0.0,808.0
3,2026-09-05,Saturday,Weekend,Grab,Delivery,Aggregator,1,843.0,252.9,590.1
4,2026-09-05,Saturday,Weekend,Phone,Dine-in,Own channel,1,608.0,0.0,608.0
5,2026-09-05,Saturday,Weekend,QR dine-in,Dine-in,Own channel,1,768.0,0.0,768.0



mart_sales_daily


,business_date,channel_name,menu_item_nk,item_name,category,size_name,units_sold,orders,item_sales_thb,net_revenue_thb
0,2026-09-04,Grab,GARLIC,Garlic Bread,side,n/a,1.0,1,109.0,76.3
1,2026-09-04,Grab,HAW-L,Hawaiian L,pizza,Large,1.0,1,369.0,258.3
2,2026-09-04,Grab,PEP-L,Pepperoni L,pizza,Large,1.0,1,379.0,265.3
3,2026-09-04,Phone,GARLIC,Garlic Bread,side,n/a,1.0,1,89.0,89.0
4,2026-09-04,Phone,HAW-L,Hawaiian L,pizza,Large,1.0,1,319.0,319.0
5,2026-09-04,QR dine-in,COKE,Coke,drink,n/a,2.0,1,70.0,70.0
6,2026-09-04,QR dine-in,PEP-L,Pepperoni L,pizza,Large,2.0,2,658.0,738.0
7,2026-09-05,Grab,COKE,Coke,drink,n/a,1.0,1,45.0,31.5
8,2026-09-05,Grab,PEP-L,Pepperoni L,pizza,Large,2.0,1,798.0,558.6
9,2026-09-05,Phone,HAW-L,Hawaiian L,pizza,Large,1.0,1,319.0,319.0



mart_option_daily


,business_date,channel_name,option_group,option_name,option_type,category,times_picked,option_revenue_thb
0,2026-09-04,QR dine-in,Cheese,Extra cheese,add,pizza,2.0,80.0
1,2026-09-04,QR dine-in,Spice,No chili,remove,pizza,1.0,0.0
2,2026-09-05,Phone,Cheese,Extra cheese,add,pizza,1.0,40.0


## 6 · The owner dashboard

The only consumer. Every tile reads the aggregate tables, never the facts, and never the sources.

ผู้ใช้มีแค่กลุ่มเดียว ทุกช่องบนแดชบอร์ดอ่านจาก aggregate table ไม่อ่านจาก fact และไม่อ่านจากต้นทาง

In [7]:
print("1 · Net revenue by channel (what the shop actually keeps)  |  รายได้สุทธิตามช่องทาง")
display(q('''
SELECT business_date, day_name, channel_name, orders, gross_sales_thb, commission_thb, net_revenue_thb,
       round(net_revenue_thb / orders, 2) AS avg_net_per_order_thb      -- ratio from additive parts
FROM agg.mart_channel_daily ORDER BY business_date, net_revenue_thb DESC
'''))

print("2 · Commission paid to platforms, and its share of gross  |  ค่าคอมมิชชันที่จ่ายให้แพลตฟอร์ม")
display(q('''
SELECT business_date, SUM(gross_sales_thb) AS gross_thb, SUM(commission_thb) AS commission_thb,
       round(100 * SUM(commission_thb) / SUM(gross_sales_thb), 1) AS commission_pct_of_gross
FROM agg.mart_channel_daily GROUP BY 1 ORDER BY 1
'''))

print("3 · Top items, both days  |  เมนูขายดี")
display(q('''
SELECT item_name, size_name, SUM(units_sold) AS units, SUM(item_sales_thb) AS item_sales_thb
FROM agg.mart_sales_daily GROUP BY ALL ORDER BY units DESC, item_sales_thb DESC LIMIT 5
'''))

print("4 · Options by group: what customers pick  |  ตัวเลือกตามกลุ่ม: ลูกค้าเลือกอะไร")
display(q('''
SELECT option_group, option_name, SUM(times_picked) AS times_picked, SUM(option_revenue_thb) AS option_revenue_thb
FROM agg.mart_option_daily GROUP BY ALL ORDER BY option_group, times_picked DESC
'''))

print("5 · Extra cheese per pizza sold (drill across two aggregates)  |  ชีสเพิ่มต่อพิซซ่า")
display(q('''
WITH pizzas AS (SELECT business_date, SUM(units_sold) AS pizzas FROM agg.mart_sales_daily
                WHERE category = 'pizza' GROUP BY 1),
     cheese AS (SELECT business_date, SUM(times_picked) AS extra_cheese, SUM(option_revenue_thb) AS addon_revenue_thb
                FROM agg.mart_option_daily WHERE option_name = 'Extra cheese' GROUP BY 1)
SELECT p.business_date, p.pizzas, c.extra_cheese, c.addon_revenue_thb,
       round(100.0 * c.extra_cheese / p.pizzas, 1) AS extra_cheese_pct
FROM pizzas p LEFT JOIN cheese c USING (business_date) ORDER BY 1
'''))

1 · Net revenue by channel (what the shop actually keeps)  |  รายได้สุทธิตามช่องทาง


,business_date,day_name,channel_name,orders,gross_sales_thb,commission_thb,net_revenue_thb,avg_net_per_order_thb
0,2026-09-04,Friday,QR dine-in,2,808.0,0.0,808.0,404.00
1,2026-09-04,Friday,Grab,2,857.0,257.1,599.9,299.95
2,2026-09-04,Friday,Phone,1,408.0,0.0,408.0,408.00
3,2026-09-05,Saturday,QR dine-in,1,768.0,0.0,768.0,768.00
4,2026-09-05,Saturday,Phone,1,608.0,0.0,608.0,608.00
5,2026-09-05,Saturday,Grab,1,843.0,252.9,590.1,590.10


2 · Commission paid to platforms, and its share of gross  |  ค่าคอมมิชชันที่จ่ายให้แพลตฟอร์ม


,business_date,gross_thb,commission_thb,commission_pct_of_gross
0,2026-09-04,2073.0,257.1,12.4
1,2026-09-05,2219.0,252.9,11.4


3 · Top items, both days  |  เมนูขายดี


,item_name,size_name,units,item_sales_thb
0,Pepperoni L,Large,7.0,2533.0
1,Coke,n/a,5.0,185.0
2,Hawaiian L,Large,3.0,1007.0
3,Garlic Bread,n/a,2.0,198.0
4,Pepperoni M,Medium,1.0,249.0


4 · Options by group: what customers pick  |  ตัวเลือกตามกลุ่ม: ลูกค้าเลือกอะไร


,option_group,option_name,times_picked,option_revenue_thb
0,Cheese,Extra cheese,3.0,120.0
1,Spice,No chili,1.0,0.0


5 · Extra cheese per pizza sold (drill across two aggregates)  |  ชีสเพิ่มต่อพิซซ่า


,business_date,pizzas,extra_cheese,addon_revenue_thb,extra_cheese_pct
0,2026-09-04,5.0,2.0,80.0,40.0
1,2026-09-05,6.0,1.0,40.0,16.7


Tile 5 is a **drill across**: each aggregate is summarized to the business date first, then the two are combined.
Joining `FCT_ORDER_LINE` to `FCT_ORDER_LINE_OPTION` row by row and summing would double-count any line with two
picks (P-1003 has extra cheese *and* no chili).

### What's out of scope, and how it comes back

| Later | Adds | Reuses |
|---|---|---|
| Stock and reorder | `FCT_INVENTORY_MOVEMENT`, `FCT_INVENTORY_COUNT`, recipes per item and per option | `DIM_DATE`, `DIM_MENU_ITEM`, `DIM_MODIFIER_OPTION` |
| Delivery times | `FCT_ORDER_EVENT` | `DIM_DATE`, `DIM_CHANNEL` |
| Labour | `FCT_SHIFT`, `DIM_STAFF` | `DIM_DATE` |
| Payments, promotions | `FCT_PAYMENT`, `FCT_ORDER_LINE_PROMOTION` | `DIM_DATE`, `DIM_CHANNEL`, `DIM_CUSTOMER`, `DIM_MENU_ITEM` |

Because the dimensions are conformed, each addition is a new star next to these two, not a redesign.

ช่องที่ 5 คือการ **drill across**: สรุปแต่ละ aggregate ให้อยู่ระดับวันทำการก่อน แล้วค่อยรวมกัน ถ้า join fact ระดับบรรทัดกับ fact ของตัวเลือกทีละแถวแล้ว sum
บรรทัดที่มีการเลือกสองตัวจะถูกนับซ้ำ (P-1003 มีทั้งชีสเพิ่มและไม่ใส่พริก)
ส่วนที่อยู่นอกขอบเขตเพิ่มกลับมาได้เป็น star ใหม่ข้าง ๆ fact สองตัวนี้ โดยใช้ conformed dimension ชุดเดิม

In [8]:
con.close()